# Movie Market Analysis  
#### Group 10
**Authors**: Kelvin Kipkorir, Raphael Muthenya, Sharon Aoko, Lucy Mutua, Charles Mutembei, Andrew Chege 

 ![Movie Market Analysis Dashboard](images/movie_studio.jpg)

## Overview  
SilverScreen Studios is preparing to enter the movie production industry and seeks a data-driven approach to guide its entry strategy. This project analyzes historical movie industry data to uncover the drivers of box office success and return on investment.  

The analysis utilizes statistical techniques—including hypothesis testing, t-tests, ANOVA, and linear regression—to draw meaningful inferences from the data. These methods will be used to assess whether factors such as release month, director influence, genre, and production budgets have statistically significant effects on movie revenue and profitability.  

By combining exploratory visualizations with formal statistical analysis, the project aims to identify high-performing genres, optimal release periods, and financially viable production strategies. The insights generated will equip SilverScreen Studios with the analytical foundation needed to make informed, profitable decisions in a competitive market.

## Business Problem  
Entering the movie production industry involves substantial financial investment and uncertainty. SilverScreen Studios requires a data-driven approach to reduce risks and optimize decision-making. Understanding the influence of factors such as directors, studios, genres, and release months on financial performance is essential to formulating a successful market entry strategy. This analysis seeks to determine what types of movies and release strategies yield the highest return on investment (ROI), and which market players and genres consistently lead in revenue generation.

## Objectives  
- Identify top-performing directors and genres, and evaluate their impact on the movie market.  
- Analyze the most successful studios and the revenue they generate.  
- Determine which types of movies generate the highest ROI using budget and gross revenue data.  
- Investigate whether the release month significantly affects movie revenue.  

## Data

The datasets used in this analysis are compiled from reputable online movie databases and stored in the `zippedData` folder. They cover various dimensions of the movie industry such as:

- **Genres**
- **Gross revenue**
- **Production budgets**
- **Directors**
- **Studios**
- **Audience and critic ratings**
- **Release dates**
- **Runtime**

The following data sources and corresponding datasets were used:

- **[Box Office Mojo](https://www.boxofficemojo.com/)**  
  `bom.movie_gross`: Contains box office gross performance data for various films.

- **[Rotten Tomatoes](https://www.rottentomatoes.com/)**  
  `rt.movie_info`: Includes Rotten Tomatoes movie metadata such as release year and genres.  
  `rt.reviews`: Features critic reviews and ratings from Rotten Tomatoes.

- **[The Movie Database (TMDB)](https://www.themoviedb.org/)**  
  `tmdb.movies`: Offers extensive metadata including movie popularity, genres, and runtime.

- **[The Numbers](https://www.the-numbers.com/)**  
  `tn.movie_budgets`: Details movie production budgets and worldwide gross earnings.
  
- **`IMDB Movies Dataset from kaggle`**
  `imdb_movies.csv`: The dataset contains genres, revenue, budget among other columns


These datasets were cleaned, joined, and analyzed using exploratory data analysis and statistical methods to uncover key insights into profitability, director performance, release timing, and genre trends within the movie market.


In [1]:
#import standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns
import statsmodels.api as sm
import zipfile
import sqlite3
import warnings

warnings.filterwarnings('ignore')
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (16, 9)

In [2]:
#reading the datasets
rt_movie_info = pd.read_csv('zippedData/rt.movie_info.tsv.gz', sep='\t',encoding='latin1')
box_office = pd.read_csv('zippedData/bom.movie_gross.csv.gz',encoding='latin1')
rt_reviews = pd.read_csv('zippedData/rt.reviews.tsv.gz', sep='\t',encoding='latin1')
tmdb_movies = pd.read_csv('zippedData/tmdb.movies.csv.gz',encoding='latin1')
budgets_df = pd.read_csv('zippedData/tn.movie_budgets.csv.gz',encoding='latin1')
#imdb_data = pd.read_csv('data/imdb_movies.csv')

Exploring the look of different datasets 

In [3]:
tmdb_movies

,Unnamed: 0,genre_ids,id,original_language,original_title,popularity,release_date,title,vote_average,vote_count
0,0,"[12, 14, 10751]",12444,en,Harry Potter and the Deathly Hallows: Part 1,33.533,2010-11-19,Harry Potter and the Deathly Hallows: Part 1,7.7,10788
1,1,"[14, 12, 16, 10751]",10191,en,How to Train Your Dragon,28.734,2010-03-26,How to Train Your Dragon,7.7,7610
2,2,"[12, 28, 878]",10138,en,Iron Man 2,28.515,2010-05-07,Iron Man 2,6.8,12368
3,3,"[16, 35, 10751]",862,en,Toy Story,28.005,1995-11-22,Toy Story,7.9,10174
4,4,"[28, 878, 12]",27205,en,Inception,27.920,2010-07-16,Inception,8.3,22186
...,...,...,...,...,...,...,...,...,...,...
26512,26512,"[27, 18]",488143,en,Laboratory Conditions,0.600,2018-10-13,Laboratory Conditions,0.0,1
26513,26513,"[18, 53]",485975,en,_EXHIBIT_84xxx_,0.600,2018-05-01,_EXHIBIT_84xxx_,0.0,1
26514,26514,"[14, 28, 12]",381231,en,The Last One,0.600,2018-10-01,The Last One,0.0,1
26515,26515,"[10751, 12, 28]",366854,en,Trailer Made,0.600,2018-06-22,Trailer Made,0.0,1


In [4]:
box_office

,title,studio,domestic_gross,foreign_gross,year
0,Toy Story 3,BV,415000000.0,652000000,2010
1,Alice in Wonderland (2010),BV,334200000.0,691300000,2010
2,Harry Potter and the Deathly Hallows Part 1,WB,296000000.0,664300000,2010
3,Inception,WB,292600000.0,535700000,2010
4,Shrek Forever After,P/DW,238700000.0,513900000,2010
...,...,...,...,...,...
3382,The Quake,Magn.,6200.0,NaN,2018
3383,Edward II (2018 re-release),FM,4800.0,NaN,2018
3384,El Pacto,Sony,2500.0,NaN,2018
3385,The Swan,Synergetic,2400.0,NaN,2018


In [5]:
rt_reviews

,id,review,rating,fresh,critic,top_critic,publisher,date
0,3,A distinctly gallows take on contemporary fina...,3/5,fresh,PJ Nabarro,0,Patrick Nabarro,"November 10, 2018"
1,3,It's an allegory in search of a meaning that n...,NaN,rotten,Annalee Newitz,0,io9.com,"May 23, 2018"
2,3,... life lived in a bubble in financial dealin...,NaN,fresh,Sean Axmaker,0,Stream on Demand,"January 4, 2018"
3,3,Continuing along a line introduced in last yea...,NaN,fresh,Daniel Kasman,0,MUBI,"November 16, 2017"
4,3,... a perverse twist on neorealism...,NaN,fresh,NaN,0,Cinema Scope,"October 12, 2017"
...,...,...,...,...,...,...,...,...
54427,2000,The real charm of this trifle is the deadpan c...,NaN,fresh,Laura Sinagra,1,Village Voice,"September 24, 2002"
54428,2000,NaN,1/5,rotten,Michael Szymanski,0,Zap2it.com,"September 21, 2005"
54429,2000,NaN,2/5,rotten,Emanuel Levy,0,EmanuelLevy.Com,"July 17, 2005"
54430,2000,NaN,2.5/5,rotten,Christopher Null,0,Filmcritic.com,"September 7, 2003"


In [6]:
budgets_df

,id,release_date,movie,production_budget,domestic_gross,worldwide_gross
0,1,"Dec 18, 2009",Avatar,"$425,000,000","$760,507,625","$2,776,345,279"
1,2,"May 20, 2011",Pirates of the Caribbean: On Stranger Tides,"$410,600,000","$241,063,875","$1,045,663,875"
2,3,"Jun 7, 2019",Dark Phoenix,"$350,000,000","$42,762,350","$149,762,350"
3,4,"May 1, 2015",Avengers: Age of Ultron,"$330,600,000","$459,005,868","$1,403,013,963"
4,5,"Dec 15, 2017",Star Wars Ep. VIII: The Last Jedi,"$317,000,000","$620,181,382","$1,316,721,747"
...,...,...,...,...,...,...
5777,78,"Dec 31, 2018",Red 11,"$7,000",$0,$0
5778,79,"Apr 2, 1999",Following,"$6,000","$48,482","$240,495"
5779,80,"Jul 13, 2005",Return to the Land of Wonders,"$5,000","$1,338","$1,338"
5780,81,"Sep 29, 2015",A Plague So Pleasant,"$1,400",$0,$0


In [7]:
imdb_data

NameError: name 'imdb_data' is not defined

# Objective 1

The dataset for this objective analysis is ***imdb_movies.csv***, an ***IMDB Movies Dataset from kaggle***. The dataset contains genres, revenue, budget among other columns, making it useful when looking into top performing genres based on their Return On Investment(ROI).

In [ ]:
#take a look at the dataset using pandas methods
print(imdb_data.info())
print(imdb_data.describe())
print(imdb_data.isnull().sum())

In [ ]:
#drop nul values
imdb_data = imdb_data.dropna(subset=['genre'])
#Formating the date column to datetime
imdb_data['date_x'] = pd.to_datetime(imdb_data['date_x'])
imdb_data.head()

#####  Splitting the genres column into individual items e.g.(['Drama', 'Action'] for easier analysis,  grouping, or manipulation since in this case the original column is stored as a single string (e.g., 'Drama, Action')

In [ ]:
imdb_data['genre_list'] = imdb_data['genre'].str.split(', ')
imdb_data['id'] = imdb_data.index + 1
imdb_data.head()

In [ ]:
#checing if the elements in the genres_list are of the correct type, list.
print(imdb_data['genre_list'].head())
print(imdb_data['genre_list'].apply(type).unique())

In [ ]:
#ensuring that the genre_list column is processed to break down genre strings into clean, individual genres.
def genre_list1(x):
    if isinstance(x, list):
        new_list = []
        for item in x:
            if isinstance(item, str):
                new_list.extend([genre.strip() for genre in item.split(',')])
            else:
                new_list.append(item)
        return new_list
    elif isinstance(x, str):
              return [genre.strip() for genre in x.split(',')]
    else:
        return []

imdb_data['genre_list1'] = imdb_data['genre_list'].apply(genre_list1)

print(imdb_data['genre_list1'].head())

In [ ]:
#analysing the performance of different genres based on Return On Investment(ROI)
imdb_data['ROI'] = (imdb_data['revenue'] - imdb_data['budget_x']) / imdb_data['budget_x']
imdb_data_exploded = imdb_data.explode('genre_list1')
genre_roi =  imdb_data_exploded.groupby('genre_list1')['ROI'].mean().reset_index()
genre_roi = genre_roi.sort_values(by='ROI', ascending=False)

print(genre_roi.head(10))

**Visualization**
### 1. Expenditure(budget) per genre

In [ ]:
genre_budget = imdb_data_exploded.groupby('genre_list1')['budget_x'].mean().reset_index()
genre_budget = genre_budget.sort_values(by='budget_x', ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=genre_budget, x='genre_list1', y='budget_x', palette='Oranges_r')
plt.xticks(rotation=45, ha='right')
plt.title('Average Expenditure (Budget) by Genre', fontsize=16)
plt.ylabel('Average Budget (USD)')
plt.xlabel('Genre')
plt.tight_layout()
plt.show()

From the visualization above, we see the TV Movie genre is most expensive to produce where as Horror requires the least cost.

### 2. Profit per genre

In [ ]:
imdb_data_exploded['profit'] = imdb_data_exploded['revenue'] - imdb_data_exploded['budget_x']
genre_profit = imdb_data_exploded.groupby('genre_list1')['profit'].mean().reset_index()
genre_profit = genre_profit.sort_values(by='profit', ascending=False)


plt.figure(figsize=(12, 6))
sns.barplot(data=genre_profit, x='genre_list1', y='profit', palette='Blues_r')
plt.title('Average Profit by Genre', fontsize=16)
plt.ylabel('Average Profit (USD)')
plt.xlabel('Genre')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

From the graph above, we see that documentary is the most profitable genre followed by TV Movie

### 3.Top Performing Genres by Average ROI

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=genre_roi, y='ROI', x='genre_list1', palette='Greens_r')
plt.title('Top Performing Genres by Average ROI', fontsize=16)
plt.ylabel('Average ROI')
plt.xlabel('Genre')
plt.tight_layout()
plt.xticks(rotation=45, ha='right') 
plt.savefig('top_genres_by_roi.png', dpi=300, bbox_inches='tight')
plt.show()

From the graph above, we see that Horror and Thriller genres are the top performers in terms of ROI. Documentary and Mystery show the lowest ROI.

## Recommendations
1. Focus on genres with a high ROI (Horror and Thriller)
    This shows that the genres deliver more return per dollar invested, even if their total revenue is not always the highest.
2. Tighten budget control, especially in mid-tier productions. Avoid over-investment in genres where high budgets do not translate to proportional profits.


# Objective 2

### Data Cleaning SQL Database

In [ ]:
with zipfile.ZipFile('zippedData/im.db.zip', 'r') as zip_ref:
    zip_ref.extractall('unzipped_db')
## Connect to the .db file
conn = sqlite3.connect('unzipped_db/im.db')
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tables)

In [ ]:
table_names = tables['name'].tolist()
for table in table_names:
    print(f"\n=== Table: {table} ===\n")
    
    columns = pd.read_sql_query(f"PRAGMA table_info({table});", conn)
    print("Columns:")
    print(columns[['name', 'type']])
    
    sample_data = pd.read_sql_query(f"SELECT * FROM {table} LIMIT 5;", conn)
    print("\nSample rows:")
    print(sample_data)
    print("-" * 60)

In [ ]:
movie_basics="""
SELECT *
FROM movie_basics
"""
movie_basics= pd.read_sql_query(movie_basics,conn)

In [ ]:
print(movie_basics.isnull().sum())
print("*****-----*****")
print(f"duplicates:{movie_basics.duplicated().sum()}")

In [ ]:
median_runtime = movie_basics["runtime_minutes"].median()
movie_basics["runtime_minutes"] = movie_basics["runtime_minutes"].fillna(median_runtime)

mode_genres = movie_basics["genres"].mode()[0]
movie_basics["genres"] = movie_basics["genres"].fillna(mode_genres)

movie_basics["original_title"] = movie_basics["original_title"].fillna("Unknown")

movie_basics.isnull().sum()

In [ ]:
directors="""
SELECT *
FROM directors
"""
directors= pd.read_sql_query(directors,conn)
print(f"Directors: {directors.isnull().sum()}")
print("*****-----*****")

movie_ratings="""
SELECT *
FROM movie_ratings
"""
movie_ratings= pd.read_sql_query(movie_ratings,conn)
print(f"Ratings: {movie_ratings.isnull().sum()}")
print("*****-----*****")

persons = pd.read_sql_query("SELECT * FROM persons", conn)
print(f"Persons: {persons.isnull().sum()}")
print("*****-----*****")

In [ ]:
movie_basics

In [ ]:
directors

## Exploratory Data Analysis
On this section we will be merging tables and relating them to our objectives.
Let's start with directors and ratings of the movies they produce to see top performing directors

In [ ]:
#mering movie_basics with movie_ratings
movies = pd.merge(movie_basics, movie_ratings, on="movie_id", how="inner")

# with directors
movies_directed = pd.merge(movies, directors, on="movie_id", how="inner")

#persons to get director names
full_data = pd.merge(movies_directed, persons, on="person_id", how="inner")

full_data = full_data[[
    'movie_id', 'primary_title', 'start_year', 'runtime_minutes', 'genres',
    'averagerating', 'numvotes', 'person_id', 'primary_name'
]]

full_data.rename(columns={'primary_name': 'director_name'}, inplace=True)
full_data

In [ ]:
duplicate_count = full_data.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

full_data = full_data.drop_duplicates(subset=['movie_id', 'person_id'])
print(f"Shape after deduplication: {full_data.shape}")

### Step 1: Filter Directors with Enough Movies and Votes
We'll filter to include only:
Directors with at least 3 movies in the dataset and movies with a minimum number of votes (numvotes >= 100) to avoid fringe cases.

In [ ]:
filtered_data = full_data[full_data['numvotes'] >= 100]

director_movie_counts = filtered_data.groupby('director_name')['movie_id'].count()

valid_directors = director_movie_counts[director_movie_counts >= 3].index

filtered_data = filtered_data[filtered_data['director_name'].isin(valid_directors)]

### Step 2: Top Directors by Average Rating
Now, let's find the top performing director. By filtering above we can have better insights based on performing directors

In [ ]:
top_directors = (
    filtered_data.groupby('director_name')
    .agg(avg_rating=('averagerating', 'mean'),
         num_movies=('movie_id', 'count'),
         total_votes=('numvotes', 'sum'))
    .sort_values(by='avg_rating', ascending=False)
    .head(10)
)

print(top_directors)

That can show how the directors perform but might not be accurate to show who truly deserves the top slot based on consistency when it comes to producing more performing movies.
To identify the top-performing directors more reliably, we use a **hybrid ranking approach** that combines two key components:

1. **Weighted Average Rating**  
   This adjusts the director's average rating based on how many votes their films received, using a formula inspired by IMDb:
   
   $$
   \text{Weighted Rating} = \frac{v}{v + m} \cdot R + \frac{m}{v + m} \cdot C
   $$
   
   Where:
   - \( R \) = director's average movie rating
   - \( v \) = total number of votes for all movies by that director(Over 150000)
   - \( m \) = minimum vote threshold (over 200000 votes)
   - \( C \) = overall average rating across all movies

   This helps discount the effect of high ratings from low-visibility films with few votes.

2. **Rating Consistency**  
   We reward directors whose movies maintain a consistent quality. Consistency is defined as the inverse of the standard deviation of their movie ratings:
   
   $$
   \text{Consistency} = \frac{1}{\text{Standard Deviation of Ratings} + 0.01}
   $$
   
   This favors directors who deliver reliably well-received films.

3. **Hybrid Score**  
   The final score is a weighted combination:
   
   $$
   \text{Hybrid Score} = 0.7 \cdot \text{Weighted Rating} + 0.3 \cdot \text{Consistency}
   $$

   This gives us a balanced view of both **overall rating strength** and **reliability**.

The top-ranked directors based on this methodology are shown below.

In [ ]:
C = filtered_data['averagerating'].mean()
m = 200000  #This favors highly voted directors

director_stats = (
    filtered_data.groupby('director_name')
    .agg(avg_rating=('averagerating', 'mean'),
         total_votes=('numvotes', 'sum'),
         num_movies=('movie_id', 'count'),
         rating_std=('averagerating', 'std'))
    .fillna(0)
)
director_stats = director_stats[
    (director_stats['num_movies'] >= 3) &
    (director_stats['total_votes'] >= 150000) &
    (director_stats['avg_rating'] >= 7.0)
]
# Weighted rating
director_stats['weighted_rating'] = (
    (director_stats['total_votes'] / (director_stats['total_votes'] + m)) * director_stats['avg_rating'] +
    (m / (director_stats['total_votes'] + m)) * C
)

# Hybrid: weighted_rating, add bonus for consistency
director_stats['consistency'] = 1 / (director_stats['rating_std'] + 0.01)
director_stats['score'] = (
    director_stats['weighted_rating'] * 0.85 +
    director_stats['consistency'] * 0.15
)

# Final sorted result
top_directors = director_stats.sort_values('score', ascending=False).head(10).reset_index()

top_directors[['director_name', 'avg_rating', 'total_votes', 'num_movies', 'weighted_rating', 'consistency', 'score']]\
    .style.format({
        'avg_rating': '{:.2f}',
        'weighted_rating': '{:.2f}',
        'consistency': '{:.2f}',
        'score': '{:.2f}',
        'total_votes': '{:,}'
    })\
    .set_caption("Top 10 Directors Ranked by Hybrid Score")

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    x='score',
    y='director_name',
    data=top_directors,
    palette='viridis'
)

plt.title('Top 10 Directors by Hybrid Score', fontsize=16)
plt.xlabel('Hybrid Score')
plt.ylabel('Director Name')
plt.tight_layout()
plt.show()

### Step 3: Director's Crowd Attention over the years
We can also us the data we filtered to check how the directors films recieved attention over the years. We will be using the (numcounts) where the more the votes the mre the popularity of their films and that can be a good inference to see which directors would be best fit for film marketing purposes.

In [ ]:
# We'll first get the yearly stats per director,filter by total votes and store them in a data frame for plotting.
director_yearly = (
    filtered_data.groupby(['director_name', 'start_year'])
    .agg(
        avg_rating=('averagerating', 'mean'),
        total_votes=('numvotes', 'sum'),
        num_movies=('movie_id', 'count')
    )
    .reset_index()
)
top5_directors = (
    director_yearly.groupby('director_name')['total_votes']
    .sum()
    .nlargest(5)
    .index
)
top5_data = director_yearly[director_yearly['director_name'].isin(top5_directors)]

plt.figure(figsize=(15, 9))
sns.barplot(
    data=top5_data,
    x='start_year',
    y='total_votes',
    hue='director_name'
)
plt.title('Total Votes Per Year - Top 5 Directors')
plt.xlabel('Year')
plt.ylabel('Total Votes')
plt.legend(title='Director', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
top5_votes = (
    director_yearly.groupby('director_name')['total_votes']
    .sum()
    .nlargest(5)
    .reset_index()
)
plt.figure(figsize=(10, 6))
sns.barplot(
    data=top5_votes,
    x='total_votes',
    y='director_name',
    palette='Blues_d'
)
plt.title('Top 5 Directors by Total Votes')
plt.xlabel('Total Votes')
plt.ylabel('Director')
plt.tight_layout()
plt.show()


In [ ]:
filtered_data = filtered_data.merge(
    movie_basics[['movie_id', 'genres']],
    on='movie_id',
    how='left'
)
print(filtered_data.columns.duplicated())

#Renaming the columns to avoid errors is key here
filtered_data.columns = ['movie_id', 'primary_title', 'start_year', 'runtime_minutes',
                         'genres_main', 'averagerating', 'numvotes',
                         'person_id', 'director_name', 'genres_y', 'genres_duplicate','genres_duplicate2']

print(filtered_data.columns)

#Genres are stored as strings.
#The data type is an object data type and so we'll use (astype(str)) to make the genre list a string then we can  work on it.
filtered_data['genre_list'] = filtered_data['genres_main'].astype(str).str.split(',')
filtered_data = filtered_data.explode('genre_list')
filtered_data['genre_list'] = filtered_data['genre_list'].str.strip()

### Director-Genre Heatmap for Top 10 Directors

This heatmap visualizes the distribution of genres for the top 10 directors based on the number of movies they’ve directed. The plot shows the number of movies each of the top directors has directed in each genre.

- **X-axis**: Genres
- **Y-axis**: Top 10 Directors
- **Color intensity**: Represents the number of movies directed by each director in each genre.
- **Annotations**: The number of movies is displayed within each cell, with values formatted to two decimal places.

This heatmap provides insights into the genres that top directors prefer or tend to direct more frequently, helping to identify genre trends across different directors.

##### Key Insights:
- The more intense the color, the greater the number of movies a director has made in that genre.
- The plot helps in identifying any directors with a specialization in particular genres, or if a director has ventured across multiple genres.

In [ ]:
top_directors = filtered_data['director_name'].value_counts().head(10).index
top_director_data = filtered_data[filtered_data['director_name'].isin(top_directors)]
director_genre_counts = top_director_data.groupby(['director_name', 'genre_list']).size().reset_index(name='count',director_genre_matrix = director_genre_counts.pivot(index='director_name', columns='genre_list', values='count')

plt.figure(figsize=(16, 10))
sns.heatmap(director_genre_matrix, cmap='YlGnBu', annot=True, fmt='.2f', linewidths=0.5)
plt.title('Top 10 Directors vs Genre Heatmap')
plt.xlabel('Genre')
plt.ylabel('Director')
plt.tight_layout()
plt.show()


# Objective 3

In [ ]:
def explore_dataframe(df, name="DataFrame"):
    print(f"\n===== {name} Overview =====")

    print("\nShape:")
    print(df.shape)

    print("\nInfo:")
    print("-" * 40)
    df.info()

    print("\nDescription:")
    print("-" * 40)
    print(df.describe(include='all'))
explore_dataframe(box_office, name="Box Office")

### Data checking and cleaning

In [ ]:
#checking for null values
missing_percentage = (box_office.isnull().sum() / len(box_office)) * 100
print(missing_percentage.sort_values(ascending=False))

In [ ]:
#cleaning columns
box_office['foreign_gross'] = pd.to_numeric(box_office['foreign_gross'], errors='coerce')
median_foreign_gross = box_office['foreign_gross'].median()

box_office.loc[:, 'foreign_gross'] = box_office['foreign_gross'].fillna(median_foreign_gross)

domestic_gross_median = box_office['domestic_gross'].median()
box_office['domestic_gross'].fillna(domestic_gross_median, inplace=True)

studio_mode = box_office['studio'].mode()[0]  
box_office['studio'].fillna(studio_mode, inplace=True)

box_office.loc[:, 'domestic_gross'] = box_office['domestic_gross'].fillna(domestic_gross_median)

box_office.loc[:, 'studio'] = box_office['studio'].fillna(studio_mode)
#total revenue
box_office['total_gross'] = box_office['domestic_gross'] + box_office['foreign_gross']

In [ ]:
missing_percentage = (box_office.isnull().sum() / len(box_office)) * 100
print(missing_percentage.sort_values(ascending=False))

In [ ]:
#check for duplicates
num_duplicates= box_office.duplicated().sum()
print(f"Number of duplicate rows: {num_duplicates}")

### EDA and Visualisation

In [ ]:
box_office.describe()

In [ ]:
#sort and select the top 10
top_10_studios = box_office.groupby('studio')['total_gross'].sum().sort_values(ascending=False).head(10).reset_index()

#simplify - divide by 100mil
top_10_studios['total_gross_hundreds_millions'] = top_10_studios['total_gross'] / 100000000

#plot
plt.figure(figsize=(12, 6))
sns.barplot(x='total_gross_hundreds_millions', y='studio', data=top_10_studios, palette='viridis')
plt.ylabel("Studio")
plt.xlabel("Total Gross (Hundreds of Millions)")
plt.title("Top 10 Studios Total Gross")
plt.tight_layout()
plt.show()

In [ ]:
top_15_foreign = box_office.groupby('studio')['foreign_gross'].sum().sort_values(ascending=False).head(15).reset_index()

top_15_foreign['foreign_gross_hundreds_millions'] = top_15_foreign['foreign_gross'] / 100000000

plt.figure(figsize=(12, 6)) 
sns.barplot(x='foreign_gross_hundreds_millions', y='studio', data=top_15_foreign, palette='plasma')
plt.ylabel("Studio") =
plt.xlabel("Foreign Gross (Hundreds of Millions)") 
plt.title("Top 15 Studios by Foreign gross")
plt.tight_layout()
plt.show()

In [ ]:
top_30_domestic = box_office.sort_values(by='domestic_gross', ascending=False).head(30)


top_30_domestic['domestic_gross_hundreds_millions'] = top_30_domestic['domestic_gross'] / 100000000

#plot
plt.figure(figsize=(12, 6))  
sns.barplot(x='domestic_gross_hundreds_millions', y='studio', data=top_30_domestic)
plt.ylabel("Studio")
plt.xlabel("Domestic Gross (Hundreds of Millions)")
plt.title("Top Studios by Domestic Gross")
plt.tight_layout()
plt.show()

### Try find a relationship between Domestic and Foreign Gross

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='domestic_gross', y='foreign_gross', data=box_office,
                color='green', label='Domestic Gross')
sns.scatterplot(x='domestic_gross', y='foreign_gross', data=box_office,
                color='red', label='Foreign Gross')

#calculate and plot the line of best fit
x = box_office['domestic_gross']
y = box_office['foreign_gross']
m, b = np.polyfit(x, y, 1)  # Calculate slope (m) and y-intercept (b)
plt.plot(x, m*x + b, color='blue', label='Line of Best Fit')  # Plot the line

plt.title('Domestic vs. Foreign Gross')
plt.xlabel('Domestic Gross')
plt.ylabel('Foreign Gross')
plt.legend()
plt.show()

There is a general positive correlation between domestic and foreign gross. This means that movies that perform well domestically tend to also perform well in foreign markets, and vice versa.
 The correlation appears to be relatively strong, as the points are clustered around the line of best fit, suggesting a clear relationship between the two variables.
 
 A limitation to this is while most movies follow the general trend, there are some outliers. Some movies have high domestic gross but relatively low foreign gross, while others have high foreign gross but lower domestic gross

### Group by 'studio' and count the number of movies

In [ ]:
#groupby and count the number of movies
studio_counts = box_office.groupby('studio')['title'].count().sort_values(ascending=False)

#top 10 studios
top_15_studios = studio_counts.head(15)
top_15_studios

### Anova Test
Now let’s perform an ANOVA (Analysis of Variance) test. This is useful when comparing more than two groups — in this case, comparing average total gross across multiple studios.This determines if there's a statistically significant difference in total gross revenue across different movie studios.

In [ ]:
top_10_studios = box_office.groupby('studio')['title'].count().sort_values(ascending=False).head(10).index
filtered_data = box_office[box_office['studio'].isin(top_10_studios)]

#group data for anova
groups = [group['total_gross'].values for name, group in filtered_data.groupby('studio')]

#test
f_stat, p_value = stats.f_oneway(*groups)

print(f"F-statistic: {f_stat}")
print(f"P-value: {p_value}"

F-statistic: 48.557719582460166

The F-statistic measures the variance between the means of the groups (studios in your case) compared to the variance within the groups.
A larger F-statistic indicates a greater difference between the group means.
In your case, the F-statistic is relatively large (48.56), suggesting there's a significant difference in the mean total_gross between the top 10 studios.

The p-value represents the probability of observing the obtained results (or more extreme results) if there were no real difference between the group means. A smaller p-value indicates stronger evidence against the null hypothesis (that there's no difference between the group means). In your case, the p-value is extremely small (close to 0), which is much lower than the typical significance level of 0.05.

### T tests
Perform a hypothesis test using your movie box office data. The test will check whether top studios  earn significantly more than other studios in terms of total_gross

In [ ]:
top_studios = ['WB', 'BV', 'Fox', 'Sony', 'LGF']  # Example list
box_office['studio'] = box_office['studio'].astype(str)
box_office['total_gross'] = pd.to_numeric(box_office['total_gross'], errors='coerce')

#two groups
top_group = box_office[box_office['studio'].isin(top_studios)]['total_gross'].dropna()
other_group = box_office[~box_office['studio'].isin(top_studios)]['total_gross'].dropna()

# Perform independent two-sample t-test
t_stat, p_value = ttest_ind(top_group, other_group, equal_var=False)

print("T-statistic:", t_stat)
print("P-value:", p_value)

if p_value < 0.05:
    print("Reject the null hypothesis: Top studios earn significantly more.")
else:
    print("Fail to reject the null hypothesis: No significant difference found.")

The null hypothesis assumes that the average total gross revenue generated by movies from the top 10 studios is essentially the same. Any observed differences are due to random chance or sampling variability.

The alternative hypothesis proposes that the average total gross revenue generated by movies from the top 10 studios is not the same. There are real, underlying differences in performance between these studios.

Reject the null hypothesis: Based on the very small p-value, you can reject the null hypothesis. This means there's strong evidence to suggest that there's a statistically significant difference in the mean total_gross between the top 10 studios.

## Recommendations

# Objective 3


##  Does the release month of a movie significantly affect its box office revenue?

---
![Cover Image](images/movie_rel_mon2.jpg)

## Dataset Overview

The dataset used is `tn.movie_budgets.csv.gz`, located in the `zippedData` folder. It contains information about over 5,700 movies, with the following columns:

- `id`: Unique identifier for each movie  
- `release_date`: The date the movie was released  
- `movie`: Title of the movie  
- `production_budget`: Budget allocated for movie production (as a string with `$`)  
- `domestic_gross`: Revenue generated domestically (as a string with `$`)  
- `worldwide_gross`: Total worldwide revenue (as a string with `$`)  

---


In [ ]:
budgets_df.info()

In [ ]:
budgets_df.describe()

In [ ]:
#converting the the text values to numbers
cols_to_clean = ['production_budget', 'domestic_gross', 'worldwide_gross']

for col in cols_to_clean:
    budgets_df[col] = budgets_df[col].replace('[\$,]', '', regex=True).astype(float)

budgets_df.info()

In [ ]:
#changing release date to date object then making two new columns i.e the month and year
budgets_df['release_date'] = pd.to_datetime(budgets_df['release_date'], errors='coerce')
budgets_df['release_year'] = budgets_df['release_date'].dt.year
budgets_df['release_month'] = budgets_df['release_date'].dt.month
budgets_df.head(20)

In [ ]:
budgets_df.info()

## Initial Visual Exploration

To understand the distribution and relationships in the data:

- Plotted histograms for `production_budget` and `worldwide_gross` to observe skewness and outliers.  
- Created a scatter plot to explore the relationship between `production_budget` and `worldwide_gross`.  
- Used a line plot to show trends in average worldwide gross revenue over the years.  

In [ ]:
#distribution of Worldwide Gross
sns.histplot(budgets_df['worldwide_gross'], kde=True)
plt.title('Distribution of Worldwide Gross')
plt.xlabel('Worldwide Gross')
plt.ylabel('Frequency')
plt.show()

From this plot, we can observe:

- The distribution of worldwide gross is heavily right-skewed (i.e., most movies make modest revenue, but a few blockbusters make a lot).
- This skewness justifies the idea of normalization or log transformation before running models or tests that assume normality.


In [ ]:
#distribution of Production Budget
sns.histplot(budgets_df['production_budget'], kde=True)
plt.title('Distribution of Production Budget')
plt.xlabel('Production Budget')
plt.ylabel('Frequency')
plt.show()

Similar to `worldwide_gross`, it shows:

- A right-skewed distribution with a few high-budget movies.
- Possible transformation (like log) could stabilize variance in later models.

In [ ]:
#Scatterplot: Production Budget vs Worldwide Gross
sns.scatterplot(data=budgets_df, x='production_budget', y='worldwide_gross')
plt.title('Production Budget vs Worldwide Gross')
plt.xlabel('Production Budget')
plt.ylabel('Worldwide Gross')
plt.show()

There appears to be a positive relationship—higher production budgets often lead to higher gross revenue.

- However, the spread may increase with budget, suggesting heteroscedasticity (variance not being constant).
- Outliers are visible—very high budget movies that earned much more or much less than expected.

In [ ]:
#Line Plot: Average Worldwide Gross by Year
avg_gross_by_year = budgets_df.groupby('release_year')['worldwide_gross'].mean()
avg_gross_by_year.plot(kind='line', marker='o')
plt.title('Average Worldwide Gross by Year')
plt.ylabel('Avg Worldwide Gross')
plt.xlabel('Release Year')
plt.grid(True)
plt.show()

In [ ]:
sns.boxplot(data=budgets_df, x='release_month', y='worldwide_gross')
plt.title('Gross Revenue by Release Month')
plt.xlabel('Release Month')
plt.ylabel('Worldwide Gross')
plt.show()

In [ ]:
sns.scatterplot(data=budgets_df, x='release_month', y='worldwide_gross')
plt.title('Gross Revenue by Release Month')
plt.xlabel('Release Month')
plt.ylabel('Worldwide Gross')
plt.show()

In [ ]:
import calendar

#Group by release_month and calculate mean worldwide_gross
monthly_avg_gross = budgets_df.groupby('release_month')['worldwide_gross'].mean()

#Sort months in calendar order
monthly_avg_gross = monthly_avg_gross.reindex(range(1, 13))

#Plot
plt.figure(figsize=(12, 6))
sns.barplot(x=monthly_avg_gross.index, y=monthly_avg_gross.values)
plt.xticks(ticks=range(0, 12), labels=[calendar.month_abbr[i+1] for i in range(12)])
plt.title('Average Worldwide Gross by Release Month')
plt.xlabel('Month')
plt.ylabel('Average Worldwide Gross')
plt.show()

In [ ]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

budgets_df['season'] = budgets_df['release_month'].apply(month_to_season)
sns.barplot(data=budgets_df, x='season', y='worldwide_gross')
plt.title('Worldwide Gross by Season')
plt.show()

## Statistical Analysis

Three statistical approaches were used to assess whether the release month has a significant effect on worldwide box office revenue:

1. One-way ANOVA test  
2. Independent t-test  
3. Linear regression modeling

Due to the right-skewed distributions of `worldwide_gross` and `production_budget`, a log transformation was applied using NumPy's `log1p()` function. This normalization helped satisfy assumptions of normality and homoscedasticity required for parametric tests.

---

In [ ]:
budgets_df['log_worldwide_gross'] = np.log1p(budgets_df['worldwide_gross'])
budgets_df['log_production_budget'] = np.log1p(budgets_df['production_budget'])
plt.figure(figsize=(14, 6))

#Original production budget distribution
plt.subplot(1, 2, 1)
sns.histplot(budgets_df['production_budget'], bins=50, kde=True, color='lightgreen')
plt.title('Original Production Budget Distribution')
plt.xlabel('Production Budget')

#Log-transformed production budget
plt.subplot(1, 2, 2)
sns.histplot(budgets_df['log_production_budget'], bins=50, kde=True, color='orange')
plt.title('Log-Transformed Production Budget')
plt.xlabel('Log(Production Budget + 1)')

plt.tight_layout()
plt.show()

In [ ]:
plt.subplot(1, 2, 1)
sns.histplot(budgets_df['worldwide_gross'], bins=50, kde=True, color='skyblue')
plt.title('Original Worldwide Gross Distribution')
plt.xlabel('Worldwide Gross')

# Log-transformed
plt.subplot(1, 2, 2)
sns.histplot(budgets_df['log_worldwide_gross'], bins=50, kde=True, color='salmon')
plt.title('Log-Transformed Worldwide Gross')
plt.xlabel('Log(Worldwide Gross + 1)')

### ANOVA Test

A one-way ANOVA was conducted to determine whether there are statistically significant differences in mean worldwide gross revenue across the twelve release months. This test is appropriate when comparing means across multiple categorical groups.

**Hypotheses:**

- **Null hypothesis (H₀):** The mean worldwide gross revenue is the same across all release months.  
- **Alternative hypothesis (H₁):** At least one release month has a significantly different mean revenue.


In [ ]:
anova_df = budgets_df[['release_month', 'log_worldwide_gross']].dropna()

groups = [group['log_worldwide_gross'].values for name, group in anova_df.groupby('release_month')]

# Perform one-way ANOVA test
stat, p = stats.f_oneway(*groups)

print(f'F-statistic: {stat:.3f}')
print(f'p-value: {p:.3f}')

**ANOVA Results:**

- **F-statistic:** 12.209  
- **p-value:** 0.000  

The p-value indicates strong evidence against the null hypothesis. Therefore, it can be concluded that average worldwide gross revenue differs significantly between at least two release months.


### Independent T-Test

An independent t-test was conducted to compare the average worldwide gross revenue between movies released in **May** and **November**. These months were chosen due to their strategic importance in the film industry calendar.

**Hypotheses:**

- **Null hypothesis (H₀):** There is no difference in average worldwide gross between May and November releases.  
- **Alternative hypothesis (H₁):** There is a difference in average worldwide gross between May and November releases.


In [ ]:
group1 = budgets_df[budgets_df['release_month'] == 5]['log_worldwide_gross']
group2 = budgets_df[budgets_df['release_month'] == 11]['log_worldwide_gross']

t_stat, p_val = stats.ttest_ind(group1, group2, nan_policy='omit')
print(f'T-statistic: {t_stat}, P-value: {p_val}')

**T-Test Results:**

- **T-statistic:** -1.07  
- **P-value:** 0.2833  

The p-value is greater than the commonly used significance level of 0.05, so the null hypothesis cannot be rejected. This indicates that there is **no statistically significant difference** in average worldwide gross revenue between movies released in May and those released in November.

The negative t-statistic suggests that May releases may have slightly higher average revenue than November releases, but the difference is not large enough to be considered statistically significant.

## Linear Regression

![Cover Image](images/linear_reg1.jpg)
---
To explore the relationship between a movie’s production budget and its box office performance, a linear regression model was used.

**Variable Roles:**

- **Independent variable (predictor):** `production_budget`  
- **Dependent variable (response):** `worldwide_gross`

In [ ]:
# Calculate correlation
corr, p_value = stats.pearsonr(budgets_df['production_budget'], budgets_df['worldwide_gross'])
print(f"Correlation: {corr:.4f}, P-value: {p_value:.4f}")

#Scatterplot: Production Budget vs Worldwide Gross
sns.scatterplot(data=budgets_df, x='production_budget', y='worldwide_gross')
plt.title('Production Budget vs Worldwide Gross')
plt.xlabel('Production Budget')
plt.ylabel('Worldwide Gross')
plt.show()

**Justification:**

The objective is to assess whether higher production budgets are associated with higher worldwide gross revenue. Here, `production_budget` serves as the input variable that may influence or predict `worldwide_gross`, making it a suitable independent variable in a regression setup.

A Pearson correlation coefficient of **0.7483** was observed between the two variables, indicating a strong positive linear relationship. This supports the use of linear regression, as there is evidence that increases in production budget are generally associated with increases in gross revenue.

In [ ]:
results = sm.OLS(endog=budgets_df['worldwide_gross'],exog=sm.add_constant(budgets_df['production_budget'])).fit()
print(results.summary())


###  Model Overview

| Metric              | Value     | Explanation                                                                 |
|---------------------|-----------|-----------------------------------------------------------------------------|
| **R-squared**       | 0.560     | About 56% of the variation in worldwide gross can be explained by production budget. |
| **Adj. R-squared**  | 0.560     | Adjusted R² remains the same here as there's only one independent variable. |
| **F-statistic**     | 7355      | A high F-value suggests the model fits significantly better than a flat line. |
| **Prob (F-statistic)** | 0.000 | Very small p-value indicates the model is statistically significant overall. |

---

### Coefficients Table

| Term                  | Coefficient   | p-value | Interpretation                                                                            |
|-----------------------|---------------|---------|-------------------------------------------------------------------------------------------|
| **Intercept (const)** | -7,286,000    | 0.000   | The baseline value when production budget is zero .  |
| **Production Budget** | 3.13          | 0.000   | For every extra dollar spent, worldwide gross increases by ~$3.13, on average.           |

---

### Regression Equation

$$
\text{Worldwide Gross} = 3.13 \times \text{Production Budget} - 7{,}286{,}000
$$


---

In [ ]:
sm.graphics.plot_fit(results, "production_budget")
plt.show()

In [ ]:
#drawing the regression line
fig, ax = plt.subplots()
budgets_df.plot.scatter(x="production_budget", y="worldwide_gross", label="Data points", ax=ax)
sm.graphics.abline_plot(model_results=results, label="Regression line", ax=ax,color="red")
ax.legend();

In [ ]:
#visualizing residuals 
fig, ax = plt.subplots()

ax.scatter(budgets_df["production_budget"], results.resid)
ax.axhline(y=0, color="black")
ax.set_xlabel("production_budget")
ax.set_ylabel("residuals");

## Log Transformation
In the original dataset, both the **production_budget** and **worldwide_gross** had **highly skewed distributions** with long right tails. This kind of skew is common in financial data due to outliers and a wide range of values.

To address this, we applied a **logarithmic transformation** using `np.log1p()`, which safely handles zero and small values by computing `log(1 + x)`:

In [ ]:
#scatterplot of the log_transformed data
sns.scatterplot(data=budgets_df, x='log_production_budget', y='log_worldwide_gross')
plt.title('Production Budget vs Worldwide Gross')
plt.xlabel('Production Budget')
plt.ylabel('Worldwide Gross')
plt.show()

### Interpretation: Log-Log Scatter Plot (log_production_budget vs log_worldwide_gross)

- Clear positive linear relationship  
  The scatter plot shows an upward trend: as `log_production_budget` increases, `log_worldwide_gross` also tends to increase. This indicates a strong positive correlation between the two variables.

- Log transformation effectiveness  
  The data is more evenly spread compared to the original scale. The transformation supports multiplicative relationships:  
  A 1% increase in budget results in an approximately proportional percentage increase in worldwide gross.

- Zero revenue points  
  A horizontal cluster near y = 0 suggests the presence of movies with zero or near-zero revenue. These might be anomalies or failed releases and could warrant further investigation or removal.

- Modeling implications  
  The transformation improves key assumptions of linear regression:  
  - Improved linearity  
  - More constant variance (homoscedasticity)  
  - Compressed skewness


In [ ]:
#modeling a linear regression model for the log-tranformed data
results = sm.OLS(endog=budgets_df['log_worldwide_gross'],exog=sm.add_constant(budgets_df['log_production_budget'])).fit()
print(results.summary())

---

### Model Performance

- **R-squared = 0.364**: About 36.4% of the variation in log(worldwide gross) is explained by log(production budget). While not extremely high, this is a reasonably strong linear relationship considering the simplicity of the model.
- **F-statistic = 3308, p-value < 0.001**: The model is **statistically significant**, indicating that the relationship is not due to random chance.
- **t-value for slope = 57.514, p < 0.001**: The production budget is a **significant predictor** of worldwide gross in the log scale.

---

### Regression Equation

The fitted model is:

$$
\log(\text{worldwide\_gross}) = 1.6575 \cdot \log(\text{production\_budget}) - 11.3219
$$


Which can be interpreted as:

> A 1% increase in the production budget is associated with an approximate **1.66% increase** in worldwide gross revenue, holding other factors constant.



In [ ]:
#visualizing the regression line 

fig, ax = plt.subplots()
budgets_df.plot.scatter(x="log_production_budget", y="log_worldwide_gross", label="Data points", ax=ax)
sm.graphics.abline_plot(model_results=results, label="Regression line", ax=ax,color="red")
ax.legend();

In [ ]:
#visualizizing residuals
fig, ax = plt.subplots()

ax.scatter(budgets_df["log_production_budget"], results.resid)
ax.axhline(y=0, color="black")
ax.set_xlabel("log_production_budget")
ax.set_ylabel("residuals");

# Conclusions 